In [ ]:
import ee
import math

ee.Initialize()

# -------------------------
# Config
# -------------------------
target_date_str = '2023-10-18'
target_date = ee.Date(target_date_str)
SEARCH_DAYS = 14

davis_lon, davis_lat = -93.53880797, 42.0535701
roi = ee.Geometry.Rectangle([
    davis_lon - 0.1, davis_lat - 0.1,
    davis_lon + 0.1, davis_lat + 0.1
])

def resample_to_10m(image):
    return image.resample('bilinear').reproject(crs='EPSG:4326', scale=10)

# -------------------------
# Helper: get closest image
# -------------------------
def add_abs_time_diff_ms(img, date):
    diff = ee.Number(img.get('system:time_start')).subtract(date.millis()).abs()
    return img.set('abs_diff_ms', diff)

def get_closest_mosaic(col, date, search_days):
    start = date.advance(-search_days, 'day')
    end   = date.advance(search_days, 'day')
    sub = (col.filterDate(start, end)
             .map(lambda im: add_abs_time_diff_ms(im, date))
             .sort('abs_diff_ms', False)) 
    # 注意 False 是降序：时间差最大的老图垫底，时间差最小的新图盖在最上层
    return ee.Image(sub.mosaic())

# -------------------------
# Sentinel-1: only images on target day
# -------------------------
s1_day = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(roi)
          .filterDate(target_date, target_date.advance(1, 'day'))
          .filter(ee.Filter.eq('instrumentMode', 'IW'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
          .select(['VV', 'VH', 'angle']))

s1_img = ee.Image(s1_day.mosaic())

# -------------------------
# Sentinel-2: prep + closest
# -------------------------
def prep_s2(img):
    qa = img.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    img = img.updateMask(mask)
    
    # 🌟 将毫秒时间戳转为“自1970年以来的天数”，并转为 Float32
    days_since_epoch = ee.Number(img.get('system:time_start')).divide(1000 * 60 * 60 * 24).floor()
    time_band = ee.Image.constant(days_since_epoch) \
                  .rename('Sentinel2_closest_datetime').toFloat() \
                  .updateMask(mask).reproject(crs='EPSG:4326', scale=10)
    
    img_resampled = resample_to_10m(img)
    
    blue  = img_resampled.select('B2').rename('Sentinel2_B2')
    green = img_resampled.select('B3').rename('Sentinel2_B3')
    red   = img_resampled.select('B4').rename('Sentinel2_B4')
    nir   = img_resampled.select('B8').rename('Sentinel2_B8')
    swir1 = img_resampled.select('B11').rename('Sentinel2_B11')
    swir2 = img_resampled.select('B12').rename('Sentinel2_B12')
    s2_b5 = img_resampled.select('B5').rename('Sentinel2_B5')
    s2_b6 = img_resampled.select('B6').rename('Sentinel2_B6')
    s2_b7 = img_resampled.select('B7').rename('Sentinel2_B7')
    s2_b8 = img_resampled.select('B8A').rename('Sentinel2_B8A')
    s2_b9 = img_resampled.select('B9').rename('Sentinel2_B9')

    ndvi = img_resampled.normalizedDifference(['B8', 'B4']).rename('Sentinel2_NDVI')
    ndmi = img_resampled.normalizedDifference(['B8', 'B11']).rename('Sentinel2_NDMI')

    out = ee.Image.cat([
        blue, green, red, nir, swir1, swir2,
        s2_b5, s2_b6, s2_b7, s2_b8, s2_b9, ndvi, ndmi
    ]).multiply(0.0001)
    
    return out.addBands(time_band).copyProperties(img, img.propertyNames()).set('system:time_start', img.get('system:time_start'))

s2_raw = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(roi)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
          .filter(ee.Filter.notNull(['system:time_start']))) # 加一层保险

s2_img = get_closest_mosaic(s2_raw.map(prep_s2), target_date, SEARCH_DAYS)

s2_time_band = s2_img.select('Sentinel2_closest_datetime')
s2_lag_days_abs = s2_time_band.subtract(target_date.millis()).abs().divide(1000 * 60 * 60 * 24)

s2_lag_band = s2_lag_days_abs.rename('s2_lag').toFloat()
optical_lag_band = s2_lag_days_abs.rename('optical_days_lag').toFloat()

# -------------------------
# Landsat L2: closest
# -------------------------
def prep_landsat(img):
    qa = img.select('QA_PIXEL')
    cloudShadowBitMask = 1 << 4
    cloudsBitMask = 1 << 3
    mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0).And(qa.bitwiseAnd(cloudsBitMask).eq(0))
    img = img.updateMask(mask)
    
    # 🌟 将毫秒时间戳转为“自1970年以来的天数”，并转为 Float32
    days_since_epoch = ee.Number(img.get('system:time_start')).divide(1000 * 60 * 60 * 24).floor()
    time_band = ee.Image.constant(days_since_epoch) \
                  .rename('Landsat_closest_datetime').toFloat() \
                  .updateMask(mask).reproject(crs='EPSG:4326', scale=10)
    
    b10 = img.select('ST_B10').rename('Landsat_B10')
    optical_bands = img.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
        ['Landsat_B2', 'Landsat_B3', 'Landsat_B4', 'Landsat_B5', 'Landsat_B6', 'Landsat_B7']
    ).multiply(0.0001)
    
    combined = resample_to_10m(b10.addBands(optical_bands))
    return combined.addBands(time_band).copyProperties(img, img.propertyNames()).set('system:time_start', img.get('system:time_start'))

l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(roi).filter(ee.Filter.notNull(['system:time_start']))
l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(roi).filter(ee.Filter.notNull(['system:time_start']))
ls_l2 = l8.merge(l9)

landsat_img = get_closest_mosaic(ls_l2.map(prep_landsat), target_date, SEARCH_DAYS)

# DSM + terrain
# -------------------------
def calculate_terrain(img):
    return ee.Terrain.products(img)

# 这样能完美保留底层的 30m 分辨率去计算坡度
# 1. 获取主 DEM (Copernicus) 并重命名高程波段为 'elevation'
copernicus = ee.ImageCollection("COPERNICUS/DEM/GLO30") \
    .select('DEM') \
    .mosaic() \
    .rename('elevation')

# 2. 获取备用 DEM (ALOS) 并重命名高程波段为 'elevation'
alos = ee.ImageCollection("JAXA/ALOS/AW3D30/V4_1") \
    .select('DSM') \
    .mosaic() \
    .rename('elevation')

# 3. 融合：用 ALOS 填补 Copernicus 缺失的地方
combined_dem = copernicus.unmask(alos)

# 4. 关键步骤：重新赋予 30 米默认投影
# (因为 mosaic 会使影像失去默认分辨率，如果不设置，计算出来的 slope 会全是 0)
dem_proj = combined_dem.setDefaultProjection(crs='EPSG:4326', scale=30)

# 5. 在无缝融合且分辨率正确的 DEM 上统一计算地形
# (替换了你原来代码中的 .map(calculate_terrain))
terrain_mosaic = ee.Terrain.products(dem_proj)

# 6. 从计算好的结果里提取波段 (注意高程波段现在叫 'elevation')
dsm = terrain_mosaic.select('elevation').rename('DSM')
slope = terrain_mosaic.select('slope')
aspect = terrain_mosaic.select('aspect')

# 7. 弧度转换
aspect_rad = aspect.multiply(math.pi / 180.0)
slope_rad = slope.multiply(math.pi / 180.0)

# 8. 数学特征
slope_export = slope.rename('Slope')
aspect_sin = aspect_rad.sin().rename('Aspect_sin')
aspect_cos = aspect_rad.cos().rename('Aspect_cos')

# 9. 加上 1e-6 防止平原地区除零错误
twi_proxy = slope_rad.tan().add(1e-6).pow(-1).log().rename('TWI_proxy')

# 10. 打包并在最后切片
static_topo = ee.Image.cat([dsm, slope_export, aspect_sin, aspect_cos, twi_proxy]).clip(roi)

# -------------------------
# Soil texture
# -------------------------
soil_img = resample_to_10m(ee.Image("OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02").select('b0').rename('Soil_Texture_USDA'))

# Land Cover (ESA WorldCover v200)
lc_img = ee.ImageCollection("ESA/WorldCover/v200").first().select('Map').rename('LandCover')

# -------------------------
# Day_sin / Day_cos
# -------------------------
doy_rad = ee.Number(target_date.getRelative('day', 'year').add(1)).divide(365.0).multiply(2 * math.pi)
day_sin = ee.Image(0).rename('Day_sin').toFloat().add(ee.Image.constant(doy_rad.sin()).toFloat())
day_cos = ee.Image(0).rename('Day_cos').toFloat().add(ee.Image.constant(doy_rad.cos()).toFloat())

# -------------------------
# Build final composite
# -------------------------
# -------------------------
# Build final composite
# -------------------------
feature_bands = [
    "VV", "VH", "angle",
    "Sentinel2_NDVI", "Sentinel2_NDMI",
    'Sentinel2_B2', 'Sentinel2_B3', 'Sentinel2_B4', 'Sentinel2_B5', 'Sentinel2_B6', 
    'Sentinel2_B7', 'Sentinel2_B8', 'Sentinel2_B8A', 'Sentinel2_B11', 'Sentinel2_B12',
    "DSM", "Slope", "Aspect_sin", "Aspect_cos", "TWI_proxy",
    "Landsat_B10", "Landsat_B2", "Landsat_B3", "Landsat_B4", "Landsat_B5", "Landsat_B6", "Landsat_B7",
    "Soil_Texture_USDA", "LandCover",
    "Sentinel2_closest_datetime", "Landsat_closest_datetime" 
]

# 1. 提取出高精度的 Double 类型时间戳波段
time_bands = (s2_img.select('Sentinel2_closest_datetime')
              .addBands(landsat_img.select('Landsat_closest_datetime')))

# 2. 将所有物理特征合并并转为 Float32（注意去掉了内部包含的时间戳防止被降精度）
physical_features = (s1_img
             .addBands(s2_img.select(s2_img.bandNames().remove('Sentinel2_closest_datetime')))
             .addBands(landsat_img.select(landsat_img.bandNames().remove('Landsat_closest_datetime'))) 
             .addBands(static_topo)
             .addBands(optical_lag_band)
             .addBands(s2_lag_band)
             .addBands(day_sin)
             .addBands(day_cos)
             .addBands(soil_img)
             .addBands(lc_img)
             .clip(roi)
             .toFloat())

# 3. 🌟 修复：使用 addBands 进行图层叠加，而不是 add
composite = physical_features.addBands(time_bands).select(feature_bands)


# -------------------------
# Export GeoTIFF
# -------------------------
task = ee.batch.Export.image.toDrive(
    image=composite,
    description=f'filename_{target_date_str}',
    folder='GEE_Exports',
    fileNamePrefix=f'filename_{target_date_str}',
    region=roi,
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e13
)
task.start()
print("Export started:", task.id)

In [ ]:
import rasterio
import pandas as pd
import numpy as np
import os
from rasterio.warp import transform as warp_transform

# ==========================================
# 1. 路径设置
# ==========================================
tif_file = r"E:.\filename_2023-10-18.tif"
pkl_file = tif_file.replace('.tif', '_pixels.pkl')

print(f"正在读取文件: {os.path.basename(tif_file)} ...")

try:
    # ==========================================
    # 2. 读取 TIF 文件与波段信息
    # ==========================================
    with rasterio.open(tif_file) as src:
        data = src.read()
        num_bands = src.count
        nodata_value = src.nodata  
        height, width = src.height, src.width
        img_transform = src.transform
        img_crs = src.crs
        
        print(f"✅ 成功读取！图像大小: {width} x {height}，共 {num_bands} 个波段。")
        print("正在进行数据重组 (Reshaping) 和坐标计算... 这可能会消耗较多内存，请耐心等待。")
        
        # 获取波段名称
        band_names = list(src.descriptions)
        if not any(band_names):
            band_names = [f"Band_{i}" for i in range(1, num_bands + 1)]
        else:
            band_names = [desc if desc else f"Band_{i+1}" for i, desc in enumerate(band_names)]

        # ------------------------------------------
        # 🌟 新增：高效生成所有像素的坐标
        # ------------------------------------------
        # 1. 生成所有像素的列号(cols)和行号(rows)网格
        cols, rows = np.meshgrid(np.arange(width), np.arange(height))
        
        # 2. 展平为一维数组 (需要与波段数据的展平顺序完全一致)
        cols_flat = cols.flatten()
        rows_flat = rows.flatten()
        
        # 3. 使用仿射变换矩阵将行列号转为空间坐标 (加上0.5代表取像素中心点)
        x_coords, y_coords = img_transform * (cols_flat + 0.5, rows_flat + 0.5)
        
        # 4. 检查坐标系：如果不是 WGS84(经纬度)，则自动进行投影转换
        if img_crs and img_crs.to_epsg() != 4326:
            print(f"⚠️ 检测到影像坐标系为 {img_crs}，正在自动转换为经纬度 (EPSG:4326)...")
            x_coords, y_coords = warp_transform(img_crs, 'EPSG:4326', x_coords, y_coords)
            print("✅ 坐标转换完成！")
            
        # 释放行列号网格以节省内存
        del cols, rows, cols_flat, rows_flat

    # ==========================================
    # 3. 展平数组并创建 DataFrame
    # ==========================================
    flattened_data = data.reshape(num_bands, -1).T
    
    # 构建 DataFrame，并把波段数据放进去
    df = pd.DataFrame(flattened_data, columns=band_names)
    
    # 🌟 新增：将经纬度作为前两列插入 DataFrame
    df.insert(0, 'longitude', x_coords)
    df.insert(1, 'latitude', y_coords)
    
    # 释放原始数组以节省内存
    del data, flattened_data, x_coords, y_coords
    
    # ==========================================
    # 4. 清洗无效像素 (NoData)
    # ==========================================
    initial_rows = len(df)
    
    if nodata_value is not None:
        # 假设只要第一波段是 nodata，通常整个像素就是黑边
        df = df[df[band_names[0]] != nodata_value]
    else:
        # 如果没有明确标记，删除全为 0 或含 NaN 的行
        df = df.dropna()
        
    final_rows = len(df)
    print(f"🧹 数据清洗：已移除 {initial_rows - final_rows} 个无效背景像素。")
    print(f"📊 最终有效像素总数（表格行数）: {final_rows} 条")
    
    # ==========================================
    # 5. 保存为 pkl 文件
    # ==========================================
    print(f"💾 正在保存至 PKL 文件...")
    df.to_pickle(pkl_file)
    print(f"🎉 保存成功！文件位置: {pkl_file}")
    
    print("\n预览前 5 行数据：")
    print(df.head())

except Exception as e:
    print(f"❌ 运行出错: {e}")

In [ ]:
import pandas as pd
import numpy as np

# 1. 统一设定目标日期
df["datetime"] = pd.to_datetime("2023-10-18")

# 2. 🌟 关键修改：加上 unit='D'，把 20035.0 这种天数转换成真实的日期时间
df["Sentinel2_closest_datetime"] = pd.to_datetime(df["Sentinel2_closest_datetime"], unit='D', errors="coerce")
df["Landsat_closest_datetime"] = pd.to_datetime(df["Landsat_closest_datetime"], unit='D', errors="coerce")

# 3. 计算 lag (目标日期与遥感影像日期的绝对天数差)
df["s2_lag"] = (df["datetime"] - df["Sentinel2_closest_datetime"]).dt.total_seconds().abs() / 86400
df["landsat_lag"] = (df["datetime"] - df["Landsat_closest_datetime"]).dt.total_seconds().abs() / 86400

# 4. 提取 Day of Year 并计算三角函数特征
doy = df["datetime"].dt.dayofyear
df["Day_sin"] = np.sin(2 * np.pi * doy / 365.0)
df["Day_cos"] = np.cos(2 * np.pi * doy / 365.0)

# 5. 计算 NDVI (直接使用 df 里原有的波段列)
df["Landsat_NDVI"] = (df["Landsat_B5"] - df["Landsat_B4"]) / (df["Landsat_B5"] + df["Landsat_B4"])
df["Landsat_NDMI"] = (df["Landsat_B5"] - df["Landsat_B6"]) / (df["Landsat_B5"] + df["Landsat_B6"])
df["Landsat_B10"] = df["Landsat_B10"] * 0.0001
    
mask_landsat_ndmi_better = (df['landsat_lag'] < df['s2_lag']) & df['Landsat_NDMI'].notna()
df.loc[mask_landsat_ndmi_better, 'NDMI_Best'] = df['Landsat_NDMI']
df.loc[mask_landsat_ndmi_better, 'NDVI_Best'] = df['Landsat_NDVI']
    
# 3. 补漏 (如果 S2 是 NaN，不管 Lag 多少，只能用 Landsat)
df['NDMI_Best'] = df['NDMI_Best'].fillna(df['Landsat_NDMI'])
df['NDMI_Best'] = df['NDVI_Best'].fillna(df['Landsat_NDVI'])

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.warp import transform, calculate_default_transform

# ==========================================
# 0. 准备工作
# ==========================================
# 假设 df_processed 是 WCM 修正后的 DataFrame
# tif_path 指向你的 Beck_KG 气候分类文件
tif_path = r".\data\Beck_KG_V1_future_0p0083.tif"

# ==========================================
# 1. 定义采样函数 (去除多余的坐标变换)
# ==========================================
def sample_climate_data_10m(df, tif_path, prefix="BeckKG"):
    """
    使用 WarpedVRT 在内存中动态将栅格重采样到 10m 分辨率，然后提取特征
    """
    df_out = df.copy()
    
    lons = df_out['longitude'].values
    lats = df_out['latitude'].values
    
    # 10米在 WGS84(经纬度) 下大约是 0.00008983 度
    target_res = 0.000089831 
    
    with rasterio.open(tif_path) as src:
        print(f"Original Raster CRS: {src.crs}")
        print(f"Original Resolution: {src.res}")
        
        # 坐标转换逻辑
        if src.crs and src.crs.to_string() not in ["EPSG:4326", "WGS84", "+init=epsg:4326"]:
            print("Transforming point coordinates to match raster CRS...")
            xs, ys = transform("EPSG:4326", src.crs, lons.tolist(), lats.tolist())
        else:
            xs, ys = lons.tolist(), lats.tolist()

        coords = list(zip(xs, ys))
        
        # 计算 10m 分辨率下的 Transform 和宽高
        vrt_transform, vrt_width, vrt_height = calculate_default_transform(
            src.crs, src.crs, src.width, src.height, *src.bounds, resolution=target_res
        )
        
        # 配置虚拟栅格 (VRT) 的参数
        vrt_options = {
            'resampling': Resampling.nearest, # 气候类别数据必须用 nearest
            'crs': src.crs,
            'transform': vrt_transform,
            'height': vrt_height,
            'width': vrt_width
        }
        
        print(f"Dynamically resampling to ~10m (Resolution: {target_res} degrees)...")
        # 使用 WarpedVRT 创建一个内存中的 10m 虚拟图层
        with WarpedVRT(src, **vrt_options) as vrt:
            print("Sampling from 10m VRT...")
            sampled = np.array(list(vrt.sample(coords)))
            
            # 处理 NoData
            nodata = vrt.nodata
            if nodata is not None:
                sampled = sampled.astype("float32")
                sampled[sampled == nodata] = np.nan
            
    # 将采样结果合并回 DataFrame
    n_bands = sampled.shape[1]
    new_cols = []
    for i in range(n_bands):
        col_name = f"{prefix}_band{i+1}"
        df_out[col_name] = sampled[:, i]
        new_cols.append(col_name)
        
    print(f"成功提取气候特征: {new_cols}")
    return df_out, new_cols

# ==========================================
# 2. 执行提取
# ==========================================
print("正在添加气候分类特征...")
df_with_climate, climate_cols = sample_climate_data_10m(
    df, 
    tif_path, 
    prefix="BeckKG"
)

# 检查一下提取结果
print(df_with_climate[climate_cols + ['latitude', 'longitude']].head())

In [ ]:
save_path = r'.\filename_2023-10-18_for_pred.pkl'
df_with_climate.to_pickle(save_path)

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn

# =========================
# Paths & Config
# =========================
MODEL_DIR = r".\model"

LE_PATH         = r".\data\label_encoders.pkl"
CATDIMS_PATH    = r".\data\cat_dims.pkl"
STANDARDIZER_PATH = r".\data\standardizer.joblib"
df = pd.read_pickle(r".\filename_2023-10-18_for_pred.pkl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# Feature Definitions (必须与训练完全一致)
# =========================
NUMERIC_COLS = [
    "angle", "VV", "VH", "VH_minus_VV",
    "s2_lag", "landsat_lag",
    "DSM", "Slope", "TWI_proxy", "Aspect_sin", "Aspect_cos",
    'Sentinel2_B2', 'Sentinel2_B3', 'Sentinel2_B4', 'Sentinel2_B5',
    'Sentinel2_B6', 'Sentinel2_B7', 'Sentinel2_B8', 'Sentinel2_B8A',
    'Sentinel2_B11', 'Sentinel2_B12',
    'Landsat_B2', 'Landsat_B3', 'Landsat_B4', 'Landsat_B5',
    'Landsat_B6', 'Landsat_B7', 'Landsat_B10',
    "NDVI_Best", "NDMI_Best", "Day_sin", "Day_cos"
]

CAT_COLS = ["BeckKG_band1", "Soil_Texture_USDA", "LandCover"]

# # 直接使用定义好的 NUMERIC_COLS 作为深度学习模型的特征顺序
cnn_ordered_cols = NUMERIC_COLS

# =========================
# Load Preprocessors & Data
# =========================
print("Loading preprocessors...")
label_encoders = joblib.load(LE_PATH)
cat_dims = joblib.load(CATDIMS_PATH)

# 从 joblib 中解包所有训练状态
pack = joblib.load(STANDARDIZER_PATH) 
trained_scaler = pack['scaler']         
trained_cols = pack['cols']             
trained_medians = pack['fillna_median'] 

def transform_only(df: pd.DataFrame, pack_dict: dict):
    df_out = df.copy()
    
    scaler = pack_dict['scaler']
    cols_exist = pack_dict['cols'] # 强制使用训练时的列顺序
    
    missing = [c for c in cols_exist if c not in df_out.columns]
    if missing:
        print(f"[WARN] Missing columns (filled with training median): {missing}")
        for m in missing:
            df_out[m] = pack_dict['fillna_median'].get(m, 0) # 缺失整列也用历史中位数补齐

    # 1. 处理时间差格式
    lag_cols = [c for c in cols_exist if 'lag' in c.lower()]
    for col in lag_cols:
        if df_out[col].dtype == 'object': 
            df_out[col] = pd.to_timedelta(df_out[col], errors='coerce').dt.total_seconds() / (24 * 3600)

    # 提取特征并转 float
    X = df_out[cols_exist].astype(np.float32)

    # 🌟 使用训练集的 Median 填充 NaN
    train_med_series = pd.Series(pack_dict['fillna_median'])
    X_filled = X.fillna(train_med_series)

    # 3. 使用训练好的 scaler 进行 transform
    X_std = scaler.transform(X_filled.values.astype(np.float32))

    df_out.loc[:, cols_exist] = X_std.astype(np.float32)
    
    return df_out, missing

# 运行专用推理函数，传入完整的 pack
df_std, skipped = transform_only(df, pack)

cols_exist = [c for c in NUMERIC_COLS if c in df_std.columns]

scaled_stats = pd.DataFrame({
    "min":  df_std[cols_exist].min(axis=0),
    "max":  df_std[cols_exist].max(axis=0),
    "mean": df_std[cols_exist].mean(axis=0),
}).sort_index()

print("\n===== AFTER SCALING (df_prepared) stats: min/max/mean =====")
print(scaled_stats.to_string())
print("\nMedian |mean|:", scaled_stats["mean"].abs().median())

In [ ]:
import os
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

# ============================================================
# 0. Config
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR = r".\model"     
OUT_DIR  = r".\prediction"   

N_FOLDS = 5
BATCH_SIZE_INFER = 4096

TRANS_PATHS = [
    os.path.join(SAVE_DIR, f"transformer_v4_fold_{i}.pth")
    for i in range(1, N_FOLDS + 1)
]

PREP_PATHS = [
    os.path.join(SAVE_DIR, f"transformer_v4_fold_{i}.pkl")
    for i in range(1, N_FOLDS + 1)
]

# ============================================================
# 1. Utility functions
# ============================================================

def encode_cats_unknown_safe(df, label_encoders, cat_cols):
    df = df.copy()
    for col in cat_cols:
        df[col] = df[col].fillna("Missing").astype(str)
        le = label_encoders[col]
        classes = set(le.classes_)
        s = df[col]
        s = s.where(s.isin(classes), other=le.classes_[0])
        df[col] = le.transform(s)
    return df

def build_modality_index_map(num_col_names):
    """
    必须与训练代码完全一致
    """
    SAR_FEATURES = {
        "angle", "VV_soil", "VH_soil", "VV", "VH", "VH_minus_VV",
    }

    OPTICAL_THERMAL_FEATURES = {
        'Sentinel2_B2', 'Sentinel2_B3', 'Sentinel2_B4', 'Sentinel2_B5', 'Sentinel2_B6', 
        'Sentinel2_B7', 'Sentinel2_B8', 'Sentinel2_B8A', 'Sentinel2_B11', 'Sentinel2_B12',
        'Landsat_B2', 'Landsat_B3', 'Landsat_B4', 'Landsat_B5','Landsat_B6', 'Landsat_B7', 'Landsat_B10', 
        "NDVI_Best", "NDMI_Best"
    }

    TEMPORAL_FEATURES = {
        "Day_sin", "Day_cos", "s2_lag", "landsat_lag"
    }

    idx_map = {
        "sar": [],
        "optical_thermal": [],
        "temporal": [],
        "static": [],
    }

    for i, col in enumerate(num_col_names):
        if col in SAR_FEATURES:
            idx_map["sar"].append(i)
        elif col in OPTICAL_THERMAL_FEATURES:
            idx_map["optical_thermal"].append(i)
        elif col in TEMPORAL_FEATURES:
            idx_map["temporal"].append(i)
        else:
            idx_map["static"].append(i)

    dynamic_idx = sorted(idx_map["sar"] + idx_map["optical_thermal"] + idx_map["temporal"])
    dynamic_mask = np.zeros(len(num_col_names), dtype=bool)
    dynamic_mask[dynamic_idx] = True
    static_mask = ~dynamic_mask

    return idx_map, dynamic_mask, static_mask


def transform_num_features(X_num, prep):
    """
    ★ 与 V4 训练代码一致：
    - sar / optical_thermal 用 PowerTransformer (Yeo-Johnson)
    - temporal 用 RobustScaler
    - static 用 QuantileTransformer
    """
    X_out = X_num.copy()
    idx_map = prep["idx_map"]

    for modality in ["sar", "optical_thermal", "temporal"]:
        idx = idx_map[modality]
        if len(idx) > 0:
            X_out[:, idx] = prep["scalers"][modality].transform(X_out[:, idx])

    static_mask = prep["static_mask"]
    if static_mask.sum() > 0:
        X_out[:, static_mask] = prep["scalers"]["static"].transform(X_out[:, static_mask])

    return X_out


# ============================================================
# 2. Model definition
#    ★ 与 V4 训练代码完全一致
# ============================================================

class MLPFeatureTokenizer(nn.Module):
    """★ V4 使用 2 层 MLP 投射，而非 V3 的单层 Linear"""
    def __init__(self, num_features, d_model):
        super().__init__()
        self.num_features = num_features
        self.feature_scales = nn.Parameter(torch.ones(num_features) * 0.5)

        self.num_projs = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1, d_model // 2),
                nn.GELU(),
                nn.Linear(d_model // 2, d_model),
                nn.LayerNorm(d_model)
            )
            for _ in range(num_features)
        ])

        self.pos_embedding = nn.Parameter(torch.randn(1, num_features, d_model) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

    def forward(self, x):
        B = x.size(0)
        scaled_input = x * F.softplus(self.feature_scales).unsqueeze(0)
        tokens = [proj(scaled_input[:, i:i+1]).unsqueeze(1) for i, proj in enumerate(self.num_projs)]
        tokens = torch.cat(tokens, dim=1)
        tokens = tokens + self.pos_embedding

        cls = self.cls_token.expand(B, -1, -1)
        return torch.cat([cls, tokens], dim=1)


class PreNormTransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout),
        )
        self.drop1 = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + self.drop1(h)
        x = x + self.ffn(self.norm2(x))
        return x


class TransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            PreNormTransformerLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class CrossAttentionFusion(nn.Module):
    """★ V4 版本: forward 只返回 h（不返回 attn_weights）"""
    def __init__(self, d_model, nhead, dropout=0.3):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, query, key_value):
        attn_out, _ = self.cross_attn(query, key_value, key_value)
        h = query + self.drop(attn_out)
        h = h + self.ffn(self.norm(h))
        return h.squeeze(1)


class EnhancedSoilTransformer_v4(nn.Module):
    """
    ★ V4 关键区别：
    - 每个模态有独立的 MLPFeatureTokenizer（不是 ModalityStream）
    - 所有模态 token 拼接后共享一个 TransformerEncoder
    - CrossAttentionFusion 返回单个 tensor
    """
    def __init__(
        self,
        num_col_names,
        cat_col_names,
        cat_dims_list,
        d_model=168,
        nhead=8,
        num_layers_per_stream=2,
        dim_feedforward=512,
        dropout=0.25,
        static_dropout=0.45,
    ):
        super().__init__()

        self.num_col_names = list(num_col_names)
        self.cat_col_names = list(cat_col_names)

        self.idx_map, self.dynamic_mask, self.static_mask = build_modality_index_map(num_col_names)
        self.dynamic_idx = sorted(
            self.idx_map["sar"] +
            self.idx_map["optical_thermal"] +
            self.idx_map["temporal"]
        )
        self.static_idx = self.idx_map["static"]

        # categorical embeddings
        self.has_cat = len(cat_dims_list) > 0
        self.embeddings = nn.ModuleList()
        self.emb_total_dim = 0
        if self.has_cat:
            for d in cat_dims_list:
                ed = int(min(16, round(np.sqrt(d)) + 1))
                self.embeddings.append(nn.Embedding(d, ed))
                self.emb_total_dim += ed

        self.has_sar = len(self.idx_map["sar"]) > 0
        self.has_opt = len(self.idx_map["optical_thermal"]) > 0
        self.has_temp = len(self.idx_map["temporal"]) > 0

        # ★ 每个模态独立的 MLPFeatureTokenizer + modality embedding
        if self.has_sar:
            self.sar_tokenizer = MLPFeatureTokenizer(len(self.idx_map["sar"]), d_model)
            self.modality_emb_sar = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        if self.has_opt:
            self.opt_tokenizer = MLPFeatureTokenizer(len(self.idx_map["optical_thermal"]), d_model)
            self.modality_emb_opt = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        if self.has_temp:
            self.temp_tokenizer = MLPFeatureTokenizer(len(self.idx_map["temporal"]), d_model)
            self.modality_emb_temp = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # ★ 共享的 dynamic encoder（不是每个 stream 各一个）
        self.shared_dynamic_encoder = TransformerEncoder(
            d_model, nhead, dim_feedforward, dropout, num_layers=num_layers_per_stream
        )

        # static projection
        static_out_dim = len(self.static_idx) + self.emb_total_dim
        self.static_proj = nn.Sequential(
            nn.Linear(static_out_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.static_dropout = nn.Dropout(static_dropout)

        # fusion
        self.fusion = CrossAttentionFusion(d_model, nhead, dropout)

        # prediction head
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(d_model // 4, 1),
        )

    def encode_dynamic(self, x_num):
        """★ tokenize 各模态 → 拼接 → 共享 encoder"""
        token_blocks = []
        if self.has_sar:
            t_sar = self.sar_tokenizer(x_num[:, self.idx_map["sar"]]) + self.modality_emb_sar
            token_blocks.append(t_sar)
        if self.has_opt:
            t_opt = self.opt_tokenizer(x_num[:, self.idx_map["optical_thermal"]]) + self.modality_emb_opt
            token_blocks.append(t_opt)
        if self.has_temp:
            t_temp = self.temp_tokenizer(x_num[:, self.idx_map["temporal"]]) + self.modality_emb_temp
            token_blocks.append(t_temp)

        all_dynamic_tokens = torch.cat(token_blocks, dim=1)
        return self.shared_dynamic_encoder(all_dynamic_tokens)

    def forward(self, x_num, x_cat):
        encoded_dynamic = self.encode_dynamic(x_num)

        static_feats = [x_num[:, self.static_idx]]
        if self.has_cat and x_cat.shape[1] > 0:
            embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
            static_feats.extend(embs)

        x_static = self.static_dropout(torch.cat(static_feats, dim=1))
        x_static_proj = self.static_proj(x_static).unsqueeze(1)

        # ★ V4 fusion 只返回一个 tensor
        fused = self.fusion(x_static_proj, encoded_dynamic)
        return self.head(fused).squeeze(1)


def build_transformer_v4_from_ckpt(ckpt, cat_dims_list):
    """
    用 checkpoint 里保存的结构参数恢复模型
    """
    model_params = ckpt["model_params"]

    return EnhancedSoilTransformer_v4(
        num_col_names=ckpt["num_col_names"],
        cat_col_names=ckpt["cat_col_names"],
        cat_dims_list=cat_dims_list,
        d_model=model_params["d_model"],
        nhead=model_params["nhead"],
        num_layers_per_stream=model_params["num_layers_per_stream"],
        dim_feedforward=model_params["dim_feedforward"],
        dropout=model_params["dropout"],
        static_dropout=0.45,
    ).to(device)


# ============================================================
# 3. Fold-wise inference
# ============================================================
@torch.no_grad()
def predict_transformer_v4_folds(
    ckpt_paths,
    prep_paths,
    df_input,
    label_encoders,
    cat_dims,
    model_name="TransformerV4",
    batch_size=4096,
):
    preds_folds = []

    for k, (ckpt_path, prep_path) in enumerate(zip(ckpt_paths, prep_paths), start=1):
        if not os.path.exists(ckpt_path):
            print(f"  ⚠️ Skip fold {k}: checkpoint not found -> {ckpt_path}")
            continue
        if not os.path.exists(prep_path):
            print(f"  ⚠️ Skip fold {k}: preprocessor not found -> {prep_path}")
            continue

        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        prep = joblib.load(prep_path)

        model = build_transformer_v4_from_ckpt(ckpt, cat_dims_list=cat_dims)
        model.load_state_dict(ckpt["model_state_dict"], strict=True)
        model.eval()

        NUMERIC_COLS = ckpt["num_col_names"]
        CAT_COLS = ckpt["cat_col_names"]

        missing_num = [c for c in NUMERIC_COLS if c not in df_input.columns]
        missing_cat = [c for c in CAT_COLS if c not in df_input.columns]
        if missing_num:
            raise ValueError(f"Missing numeric cols for fold {k}: {missing_num}")
        if missing_cat:
            raise ValueError(f"Missing categorical cols for fold {k}: {missing_cat}")

        df_fold = df_input.copy()

        # 类别安全编码
        df_enc = encode_cats_unknown_safe(df_fold, label_encoders, CAT_COLS)

        X_cat = df_enc[CAT_COLS].values.astype(np.int64)
        X_cat_t = torch.from_numpy(X_cat)

        # 当前 fold 专属数值预处理
        X_num_raw = df_enc[NUMERIC_COLS].values.astype(np.float32)
        X_num_scaled = transform_num_features(X_num_raw, prep).astype(np.float32)
        X_num_t = torch.from_numpy(X_num_scaled)

        n = X_num_t.shape[0]
        preds = np.zeros(n, dtype=np.float32)

        for i in range(0, n, batch_size):
            j = min(i + batch_size, n)
            xb_num = X_num_t[i:j].to(device)
            xb_cat = X_cat_t[i:j].to(device)
            preds[i:j] = model(xb_num, xb_cat).cpu().numpy().astype(np.float32)

        preds_folds.append(preds)
        print(f"  ✅ {model_name} fold {k} done | mean={preds.mean():.4f}")

    if len(preds_folds) == 0:
        raise FileNotFoundError("No valid transformer_v4 fold files were found.")

    return np.stack(preds_folds, axis=0).mean(axis=0)


# ============================================================
# 4. Main inference entry
# ============================================================
def run_transformer_v4_inference(df_std, save_output=True, output_name="prediction_transformer_v4.csv"):
    if not os.path.exists(LE_PATH): raise FileNotFoundError(f"label_encoders.pkl not found: {LE_PATH}")
    if not os.path.exists(CATDIMS_PATH): raise FileNotFoundError(f"cat_dims.pkl not found: {CATDIMS_PATH}")

    first_ckpt = None
    for p in TRANS_PATHS:
        if os.path.exists(p):
            first_ckpt = torch.load(p, map_location=device, weights_only=False)
            break

    if first_ckpt is None:
        raise FileNotFoundError("No transformer_v4 checkpoint found.")

    NUMERIC_COLS = first_ckpt["num_col_names"]
    CAT_COLS = first_ckpt["cat_col_names"]

    print(f"\n[Transformer V4] Input columns: num={len(NUMERIC_COLS)}, cat={len(CAT_COLS)}")
    print("\n--- Running Modality-Aware Transformer V4 Inference ---")

    pred_transformer = predict_transformer_v4_folds(
        ckpt_paths=TRANS_PATHS,
        prep_paths=PREP_PATHS,
        df_input=df_std,
        label_encoders=label_encoders,
        cat_dims=cat_dims,
        model_name="TransformerV4",
        batch_size=BATCH_SIZE_INFER,
    )

    mn, mx, avg = pred_transformer.min(), pred_transformer.max(), pred_transformer.mean()
    ok = "✅" if -0.1 < avg < 0.6 else "❌ ABNORMAL"
    print(f"\n[Sanity Check] TransformerV4: mean={avg:.4f} | min={mn:.4f} | max={mx:.4f}  {ok}")

    df_out = df_std.copy()
    df_out["prediction"] = pred_transformer

    if save_output:
        os.makedirs(OUT_DIR, exist_ok=True)
        base_cols = ["datetime", "longitude", "latitude"]
        save_cols = [c for c in base_cols if c in df_out.columns] + ["prediction"]
        out_pkl = os.path.join(OUT_DIR, output_name)
        df_out[save_cols].to_pickle(out_pkl)
        print(f"\n✅ Transformer V4 predictions successfully saved to: {out_pkl}")

    return df_out

# ============================================================
# 5. Example usage
# ============================================================
if __name__ == "__main__":
    df_pred = run_transformer_v4_inference(
        df_std=df_std,
        save_output=True,
        output_name=r".\prediction.pkl"
    )

NameError: name 'df_std' is not defined

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS

file_path = r".\prediction.pkl"

# 导入为 pandas DataFrame
prediction_df = pd.read_pickle(file_path)

# 1) 准备经纬度轴（排序很关键）
lats = np.sort(prediction_df["latitude"].unique())[::-1]   # 纬度从北到南（降序）
lons = np.sort(prediction_df["longitude"].unique())        # 经度从西到东（升序）

# 2) pivot 成规则网格（自动对齐 lon/lat）
grid = (
    prediction_df
    .pivot_table(index="latitude", columns="longitude", values="prediction", aggfunc="mean")
    .reindex(index=lats, columns=lons)
)

prediction_array = grid.to_numpy().astype(np.float32)

# 3) 分辨率（用 median diff 更稳）
xres = float(np.median(np.diff(lons)))
yres = float(np.median(np.diff(lats)))  # 注意：lats 是降序，所以 diff 通常是负的
yres = abs(yres)

# 4) transform：west=min_lon, north=max_lat
transform = from_origin(lons.min(), lats.max(), xres, yres)

# 5) 写 GeoTIFF
out_tif = r".\prediction.tif"
with rasterio.open(
    out_tif,
    "w",
    driver="GTiff",
    height=prediction_array.shape[0],
    width=prediction_array.shape[1],
    count=1,
    dtype="float32",
    crs=CRS.from_epsg(4326), 
    transform=transform,
    nodata=np.nan
) as dst:
    dst.write(prediction_array, 1)

print("✅ Saved:", out_tif)
print("Lon range:", lons.min(), lons.max())
print("Lat range:", lats.min(), lats.max())